# Storage 30/60 — value and hedge, at 0 % and 10 %

A seasonal store that fills in 30 days and empties in 60. Two questions only:

1. **What is it worth**, with no funding cost and with gas funded at 10 %?
2. **What do you hedge it with**, in each case?

30/60 divides cleanly — 60 inventory clips, 2 a day in and 1 a day out — so none of the
grid-sizing care that 30/65 needs applies here. `Storage_30_65.ipynb` covers that; this
notebook stays out of it.

In [ ]:
import os, warnings

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import storage_model as sm

pd.set_option("display.width", 200, "display.max_columns", 50)
plt.rcParams.update({"figure.figsize": (12, 3.4), "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})
warnings.filterwarnings("ignore", category=FutureWarning)

if not hasattr(sm, "run_valuation"):
    raise RuntimeError("stale kernel — restart it (Kernel > Restart Kernel and Run All).")

SMOKE = os.environ.get("STORAGE_NOTEBOOK_SMOKE") == "1"
print(f"ready — storage_model from {sm.__file__}")

## 1. The deal

`FUNDING_RATE` is the second case: gas bought in summer has to be paid for and carried until
it is sold in winter, and 10 % a year is the cost of that money. In the model it is
`discount_rate`, applied to every day's cash flow.

Two things to be clear about. Storage pays on injection and receives on withdrawal, so it has
no single funding *direction* — `borrow_rate`/`invest_rate` are refused for it, and a single
rate is the supported route. And the model settles cash on the day gas moves; a contract
paying later would need its own discount curve.

In [ ]:
WDR_MWH_DAY   = 10_000.0                # maximum withdrawal, MWh/day
INJ_MWH_DAY   = 20_000.0                # maximum injection — twice as fast
CAPACITY      = 600_000.0               # = 60 days out, or 30 days in

START, END    = "2027-01-01", "2027-12-31"
VAL_DATE      = pd.Timestamp("2026-06-01")
VOL, SMR      = 0.50, 1.0
FUNDING_RATE  = 0.10                    # the second case; the first is always 0
N_P           = 12 if SMOKE else 25     # price-tree half-width

N_STATES = int(round(CAPACITY / WDR_MWH_DAY))          # 60 clips of 10,000 MWh
INJ_RATE = int(round(INJ_MWH_DAY / WDR_MWH_DAY))       # 2 clips/day
WDR_RATE = 1                                           # 1 clip/day
assert N_STATES / INJ_RATE == 30.0 and N_STATES / WDR_RATE == 60.0, "grid cannot express 30/60"

print(f"{CAPACITY:,.0f} MWh, {START} .. {END}, valued {VAL_DATE:%Y-%m-%d}")
print(f"  inject   {INJ_MWH_DAY:>8,.0f} MWh/day -> 30 days to fill")
print(f"  withdraw {WDR_MWH_DAY:>8,.0f} MWh/day -> 60 days to empty")
print(f"  grid: {N_STATES} clips of {CAPACITY/N_STATES:,.0f} MWh, "
      f"{INJ_RATE} in / {WDR_RATE} out per day")

# A plain seasonal curve: dear at each year end, cheap mid-summer.
_span = pd.date_range("2026-01-01", "2029-06-30", freq="D")
CURVE = pd.Series(25.0 + 6.0 * np.cos(2 * np.pi * (_span.dayofyear.values - 1) / 365.25),
                  index=_span)
_win = CURVE.loc[START:END]
fig, ax = plt.subplots()
ax.plot(_win.index, _win.values, color="black", lw=1.4)
ax.axhline(_win.mean(), color="tab:red", ls="--", lw=1, label=f"mean {_win.mean():.2f}")
ax.set_ylabel("EUR/MWh"); ax.legend(fontsize=8)
ax.set_title(f"Forward curve, {START} .. {END}")
plt.tight_layout(); plt.show()
print(f"curve: {_win.min():.2f} in summer to {_win.max():.2f} at the year ends — "
      f"a {_win.max()-_win.min():.2f} EUR/MWh spread to store into")

## 2. What it is worth

The same deal priced twice. `intrinsic` is the schedule you could fix today against the
forward curve; `extrinsic` is what re-optimising as prices move adds. Both are present values
at the valuation date, and per MWh of **capacity** — storage cycles, so value per net MWh
moved is undefined and the model refuses to report it.

In [ ]:
def price(rate, n_p=None, run_intrinsic=True):
    model, result = sm.run_valuation(None, dict(
        product_type="storage", valDate=VAL_DATE, storageStart=START, storageEnd=END,
        capacity_mwh=CAPACITY, daily_max=INJ_MWH_DAY, clips_per_day=INJ_RATE,
        inj_rate=INJ_RATE, wdr_rate=WDR_RATE,
        initial_inv_clips=0, terminal_inv_clips=0, inj_cost=0.0, wdr_cost=0.0,
        vol=VOL, sMR=SMR, n_p_full=N_P if n_p is None else n_p,
        run_intrinsic=run_intrinsic, discount_rate=rate, daily_curve=CURVE))
    n = model.n_t
    moved = model.prob[:n] * model.strat[:n] * model.v_step
    injected = np.clip(moved, 0, None).sum(axis=(1, 2))
    withdrawn = -np.clip(moved, None, 0).sum(axis=(1, 2))
    value = float(model.v[0, model.n_p, model.initial_state])
    # sum(DF * delta * F) reprices the deal; there is no cost leg here, both costs being 0.
    reprice = float(np.dot(model.d_curve[:n] * np.asarray(model.delta[:n]),
                           np.asarray(model.fwd)[:n]))
    return model, result, dict(
        value=value, injected=injected, withdrawn=withdrawn,
        dates=pd.DatetimeIndex(model.date_span)[:n],
        invariant=abs(reprice - value) / max(abs(value), 1.0))


RUNS = {rate: price(rate) for rate in (0.0, FUNDING_RATE)}

_rows = []
for _rate, (_m, _r, _d) in RUNS.items():
    _rows.append({"funding rate": _rate, "value EUR": _d["value"],
                  "EUR/MWh capacity": _d["value"] / CAPACITY,
                  "intrinsic": _r["intrinsic"], "extrinsic": _r["extrinsic"],
                  "MWh cycled": _d["injected"].sum(),
                  "turns": _d["injected"].sum() / CAPACITY,
                  "invariant": _d["invariant"]})
_val = pd.DataFrame(_rows)
display(_val.style.hide(axis="index").format({
    "funding rate": "{:.0%}", "value EUR": "{:,.0f}", "EUR/MWh capacity": "{:.4f}",
    "intrinsic": "{:.4f}", "extrinsic": "{:.4f}", "MWh cycled": "{:,.0f}",
    "turns": "{:.2f}", "invariant": "{:.0e}"})
    .set_caption(f"30/60 storage, {CAPACITY:,.0f} MWh, vol {VOL:.0%}, "
                 f"valued {VAL_DATE:%Y-%m-%d} — present values"))

_free, _funded = _val.iloc[0], _val.iloc[1]
print(f"funding at {FUNDING_RATE:.0%} costs "
      f"{_free['value EUR'] - _funded['value EUR']:,.0f} EUR, "
      f"{1 - _funded['value EUR']/_free['value EUR']:.1%} of the value.")

# Which part of the strategy responds? The deterministic core cycle is one fill
# and one empty; the stochastic run adds optional cycling on top of it.
_rates = [0.0, FUNDING_RATE] if SMOKE else [0.0, 0.10, 0.25, 0.50, 1.00, 1.50, 2.00]
_core = pd.DataFrame([{"funding rate": _r,
                       "core cycle MWh": price(_r, n_p=0, run_intrinsic=False)[2]["injected"].sum()}
                      for _r in _rates])
_core["of capacity"] = _core["core cycle MWh"] / CAPACITY
display(_core.style.hide(axis="index").format({
    "funding rate": "{:.0%}", "core cycle MWh": "{:,.0f}", "of capacity": "{:.2f}"})
    .set_caption("Deterministic volume against funding cost — the seasonal round trip is "
                 "far enough in the money that carry does not touch it"))

_carry = _win.min() * FUNDING_RATE * 5 / 12
print(f"carry on summer gas at {FUNDING_RATE:.0%} held five months is about "
      f"{_carry:.2f} EUR/MWh, against a {_win.max()-_win.min():.2f} EUR/MWh seasonal spread,")
print(f"so the core cycle is unchanged until funding is an order of magnitude higher.")
_v0, _vr = RUNS[0.0][2]["injected"].sum(), RUNS[FUNDING_RATE][2]["injected"].sum()
print(f"what does respond is the optional cycling on top: {_v0:,.0f} -> {_vr:,.0f} MWh, "
      f"{_vr/_v0 - 1:+.1%}.")
print(f"so at these rates funding is close to a haircut on the same strategy; it changes "
      f"what you do only as it approaches the spread.")

## 3. What it does

Expected inventory, and the daily movement behind it. Injection runs at twice the withdrawal
rate, so the store fills in a third of the time it takes to empty — the asymmetry is visible
as the slope.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 5.2), sharex=True)
for (_rate, (_m, _r, _d)), _c in zip(RUNS.items(), ("tab:blue", "tab:purple")):
    _inv = np.cumsum(_d["injected"] - _d["withdrawn"])
    ax[0].plot(_d["dates"], _inv, color=_c, lw=1.4, label=f"funding {_rate:.0%}")
ax[0].axhline(CAPACITY, color="tab:red", ls="--", lw=1, label=f"capacity {CAPACITY:,.0f}")
ax[0].set_ylabel("MWh in store"); ax[0].legend(fontsize=8)
ax[0].set_title("Expected inventory — fills in 30 days, empties in 60")

_d0 = RUNS[0.0][2]
ax[1].bar(_d0["dates"], _d0["injected"], width=1.0, color="tab:green", label="injected")
ax[1].bar(_d0["dates"], -_d0["withdrawn"], width=1.0, color="tab:orange", label="withdrawn")
ax[1].axhline(INJ_MWH_DAY, color="tab:green", ls="--", lw=.9)
ax[1].axhline(-WDR_MWH_DAY, color="tab:orange", ls="--", lw=.9)
ax[1].axhline(0, color="black", lw=.8)
ax[1].set_ylabel("MWh/day, at 0 %"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

for _rate, (_m, _r, _d) in RUNS.items():
    _inv = np.cumsum(_d["injected"] - _d["withdrawn"])
    print(f"  {_rate:>4.0%}: peak inventory {_inv.max():>9,.0f} MWh "
          f"({_inv.max()/CAPACITY:5.1%} full), ends at {_inv[-1]:,.0f}, "
          f"peak rates {_d['injected'].max():,.0f} in / {_d['withdrawn'].max():,.0f} out")

## 4. The hedge

`delta` is the MWh of each month's forward you would trade to hedge that month's price risk —
negative to buy, positive to sell. It is **not** the gas: `physical` is what actually moves.

For a store the difference is stark. Physical nets to **zero** over the deal, because
everything injected is withdrawn. The hedge does not, because you buy summer forwards and
sell winter ones at different prices, and that residual is the spread you are actually long.

`delta_pv` is the same hedge tailed by the discount factor, for use against **margined
futures**, where variation margin moves today while the gas settles at delivery.

In [ ]:
def monthly(rate):
    _m, _r, _d = RUNS[rate]
    n = _m.n_t
    frame = pd.DataFrame({
        "physical": np.asarray(_m.exp_ex[:n]),
        "delta": np.asarray(_m.delta[:n]),
        "delta_pv": np.asarray(_m.delta_pv[:n])},
        index=pd.DatetimeIndex(_m.date_span)[:n])
    return frame.loc[START:END].resample("MS").sum()


_m0, _mr = monthly(0.0), monthly(FUNDING_RATE)
_hedge = pd.DataFrame({
    ("physical MWh", "0%"): _m0["physical"], ("physical MWh", f"{FUNDING_RATE:.0%}"): _mr["physical"],
    ("delta MWh", "0%"): _m0["delta"], ("delta MWh", f"{FUNDING_RATE:.0%}"): _mr["delta"],
    ("delta_pv MWh", f"{FUNDING_RATE:.0%}"): _mr["delta_pv"]})
_hedge.columns = pd.MultiIndex.from_tuples(_hedge.columns)
_hedge.index = [f"{d:%b}" for d in _hedge.index]
_hedge.loc["TOTAL"] = _hedge.sum()
display(_hedge.style.format("{:,.0f}")
        .set_properties(subset=pd.IndexSlice["TOTAL", :], **{"font-weight": "bold"})
        .set_caption("Monthly hedge — negative buys the forward, positive sells it"))

fig, ax = plt.subplots()
_x = np.arange(len(_hedge) - 1)
ax.bar(_x - 0.2, _hedge["delta MWh"]["0%"][:-1], 0.4, color="tab:blue", label="delta, 0 %")
ax.bar(_x + 0.2, _hedge["delta MWh"][f"{FUNDING_RATE:.0%}"][:-1], 0.4, color="tab:purple",
       label=f"delta, {FUNDING_RATE:.0%}")
ax.axhline(0, color="black", lw=.8)
ax.set_xticks(_x); ax.set_xticklabels(_hedge.index[:-1])
ax.set_ylabel("MWh of forward"); ax.legend(fontsize=8)
ax.set_title("Buy the summer, sell the winter — that is the hedge")
plt.tight_layout(); plt.show()

_pt, _dt = _hedge.loc["TOTAL", ("physical MWh", "0%")], _hedge.loc["TOTAL", ("delta MWh", "0%")]
print(f"physical nets to {_pt:,.0f} MWh — a store gives back everything it takes.")
print(f"delta nets to {_dt:,.0f} MWh, which is the point: the hedge is a summer/winter")
print(f"spread, not a volume. Buying and selling are in different months at different prices.")
_tail = _hedge.loc["TOTAL", ("delta_pv MWh", f"{FUNDING_RATE:.0%}")]
print(f"tailed for margined futures the book is {_tail:,.0f} MWh instead.")
for _rate, (_m, _r, _d) in RUNS.items():
    print(f"  repricing check at {_rate:.0%}: sum(DF x delta x F) matches the value "
          f"to {_d['invariant']:.0e}")

## Traps

- **`physical` and `delta` answer different questions.** How much gas, versus how much price
  risk. For a store they are barely related: physical nets to zero, the hedge does not.
- **Funding is not a haircut.** At 10 % the store both earns less *and* cycles less, because
  the marginal round trip stops covering the carry. §2 shows the volume falling with the rate.
- **Value is per MWh of capacity here**, stated on the table. Per *net* MWh moved is
  undefined for a cycling deal, which is why `profiled()` raises rather than returning zero.
- **One rate, one direction.** Storage pays and receives, so `borrow_rate`/`invest_rate` are
  refused; a single `discount_rate` or an explicit `d_curve` is the route.
- **Rates are maximums.** 30/60 caps the daily volume; it does not mean 30 injection days.